In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import datetime

In [2]:
f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")

f_cleaned['Date'] = pd.to_datetime(f_cleaned['Date'], format = '%Y-%m-%d')
m_cleaned['Date'] = pd.to_datetime(m_cleaned['Date'], format = '%Y-%m-%d')
f_sorted = f_cleaned.sort_values(['Name', 'Year'])
m_sorted = m_cleaned.sort_values(['Name', 'Year'])


Time from first competition to end of year in consideration

In [3]:
f_sorted['TimeCompeting'] = pd.to_datetime(f_sorted['Year'].astype(str) + '-12-31') - f_sorted.groupby('Name')['Date'].transform('min')
f_sorted['TimeCompeting'] = f_sorted['TimeCompeting'].dt.days


In [4]:
bomb_squat = (
    (f_sorted['Squat1Kg'] < 0) &
    (f_sorted['Squat2Kg'] < 0) &
    (f_sorted['Squat3Kg'] < 0)
)

bomb_bench = (
    (f_sorted['Bench1Kg'] < 0) &
    (f_sorted['Bench2Kg'] < 0) &
    (f_sorted['Bench3Kg'] < 0)
)

bomb_deadlift = (
    (f_sorted['Deadlift1Kg'] < 0) &
    (f_sorted['Deadlift2Kg'] < 0) &
    (f_sorted['Deadlift3Kg'] < 0)
)

f_sorted['BombOut'] = bomb_squat | bomb_bench | bomb_deadlift

Number of times a lifter has bombed out. <br>
Number of meets since last bombout, capped at 999. If lifter has never bombed out this is set to the maximum cap of 999.

In [8]:
f_sorted['NumberOfBombOuts'] = f_sorted.groupby('Name')['BombOut'].cumsum()
f_sorted['MeetsSinceLastBombOut'] = (f_sorted.groupby(['Name', 'NumberOfBombOuts']).cumcount())
f_sorted.loc[f_sorted['MeetsSinceLastBombOut']> config.CAP_MEETS_SINCE_BOMBOUT, 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT
f_sorted.loc[(f_sorted['NumberOfBombOuts'] ==0 ), 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT

Index(['Name', 'Sex', 'Division', 'BodyweightKg', 'WeightClassKg',
       'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'TotalKg', 'Date',
       'Sanctioned', 'Equipment', 'Event', 'Squat1Kg', 'Squat2Kg', 'Squat3Kg',
       'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Deadlift1Kg', 'Deadlift2Kg',
       'Deadlift3Kg', 'Year', 'TimeCompeting', 'BombOut'],
      dtype='object')